# SQL Joins and Window Functions — the Modern DS Filter

If a data-screen has one "hard" question, it's a window function. The syntax trips people because it looks like aggregation but **doesn't collapse rows**. Below, every frame of the theory runs on real data — rank, lag, running totals, sessionization — the exact four drills screens pull from.

**The mental model to say first:** *"A window function computes over a sliding frame of rows defined by `OVER (PARTITION BY ... ORDER BY ...)`, and every input row keeps its identity."* Aggregation shrinks; windows decorate.

## Setup — an events table like a real product's

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
conn.executescript('''
CREATE TABLE events (
  user_id TEXT, ts TIMESTAMP, event TEXT, revenue REAL);
CREATE TABLE users (
  user_id TEXT, plan TEXT, signup_date DATE, country TEXT);
''')

rows = [
 ("u1","2026-01-03 09:15","login",None), ("u1","2026-01-03 09:20","view",None),
 ("u1","2026-01-03 10:05","buy",45.0),   ("u1","2026-01-03 10:07","view",None),
 ("u1","2026-02-11 21:00","login",None), ("u1","2026-02-11 21:10","buy",80.0),
 ("u2","2026-01-05 14:00","login",None), ("u2","2026-01-05 14:30","buy",20.0),
 ("u2","2026-03-02 08:00","login",None), ("u2","2026-03-02 08:05","buy",120.0),
 ("u3","2026-01-20 11:00","login",None), ("u3","2026-01-20 11:02","view",None),
 ("u3","2026-02-01 12:00","login",None), ("u3","2026-02-01 12:30","buy",35.0),
 ("u4","2026-01-08 16:00","login",None), ("u4","2026-01-08 16:10","buy",60.0),
 ("u4","2026-01-09 10:00","login",None), ("u4","2026-01-09 10:01","view",None),
]
users = [("u1","pro","2025-11-01","IN"),("u2","free","2025-12-15","US"),
         ("u3","pro","2026-01-01","DE"),("u4","free","2026-01-05","US")]
conn.executemany("INSERT INTO events VALUES (?,?,?,?)", rows)
conn.executemany("INSERT INTO users VALUES (?,?,?,?)", users)
conn.commit()

def q(sql): return pd.read_sql_query(sql, conn)

q("SELECT * FROM events ORDER BY user_id, ts LIMIT 6")

,user_id,ts,event,revenue
0,u1,2026-01-03 09:15,login,NaN
1,u1,2026-01-03 09:20,view,NaN
2,u1,2026-01-03 10:05,buy,45.0
3,u1,2026-01-03 10:07,view,NaN
4,u1,2026-02-11 21:00,login,NaN
5,u1,2026-02-11 21:10,buy,80.0


## 1. ROW_NUMBER vs RANK vs DENSE_RANK — the #1 Question

The three differ **only in tie handling**, and the interview question is always "which one and why":

- `ROW_NUMBER` — 1,2,3,4 — arbitrary-but-deterministic with a tiebreaker; dedup pick.
- `RANK` — 1,2,2,4 — ties share a rank, gaps follow.
- `DENSE_RANK` — 1,2,2,3 — ties share, no gaps; "Nth distinct value" questions.

In [2]:
q('''
SELECT user_id, revenue,
       ROW_NUMBER() OVER w AS rn,
       RANK()       OVER w AS rnk,
       DENSE_RANK() OVER w AS drnk
FROM events
WHERE event = 'buy'
WINDOW w AS (PARTITION BY user_id ORDER BY revenue DESC)
ORDER BY user_id, rn
''')

,user_id,revenue,rn,rnk,drnk
0,u1,80.0,1,1,1
1,u1,45.0,2,2,2
2,u2,120.0,1,1,1
3,u2,20.0,2,2,2
4,u3,35.0,1,1,1
5,u4,60.0,1,1,1


**The follow-up that gets asked:** *"get each user's biggest purchase"* → filter `WHERE rn = 1`. Note this is where the subtlety lives: **you cannot put `WHERE rn = 1` in the same query level** — window functions evaluate after WHERE, so it needs a subquery/CTE. Writing that correctly is the actual bar:

In [3]:
q('''
WITH ranked AS (
  SELECT user_id, revenue,
         ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY revenue DESC) AS rn
  FROM events WHERE event = 'buy'
)
SELECT user_id, revenue AS biggest_purchase
FROM ranked WHERE rn = 1
ORDER BY biggest_purchase DESC
''')

,user_id,biggest_purchase
0,u2,120.0
1,u1,80.0
2,u4,60.0
3,u3,35.0


## 2. LAG / LEAD — Time-Shifted Comparison

*Time between consecutive purchases per user* — the retention/cohort primitive. `LAG(col, n)` pulls from n rows back **within the partition**; missing rows are NULL, so `COALESCE` or filter.

In [4]:
q('''
SELECT user_id, ts, revenue,
       LAG(ts) OVER (PARTITION BY user_id ORDER BY ts) AS prev_ts,
       ROUND(
         (julianday(ts) - julianday(LAG(ts) OVER (PARTITION BY user_id ORDER BY ts)))
       , 1) AS days_since_prev_purchase
FROM events
WHERE event = 'buy'
ORDER BY user_id, ts
''')

,user_id,ts,revenue,prev_ts,days_since_prev_purchase
0,u1,2026-01-03 10:05,45.0,NaN,NaN
1,u1,2026-02-11 21:10,80.0,2026-01-03 10:05,39.5
2,u2,2026-01-05 14:30,20.0,NaN,NaN
3,u2,2026-03-02 08:05,120.0,2026-01-05 14:30,55.7
4,u3,2026-02-01 12:30,35.0,NaN,NaN
5,u4,2026-01-08 16:10,60.0,NaN,NaN


**Say:** *"day-over-day / month-over-month growth is LAG: `(curr - LAG(curr)) / LAG(curr)` — and NULLs from the first row are your 'no prior period' rows, handle them explicitly."*

## 3. Running Totals and Moving Averages — the Frame Clause

`ROWS BETWEEN` defines the sliding frame. Two forms screens ask: cumulative (`UNBOUNDED PRECEDING TO CURRENT ROW`) and trailing-3 (`2 PRECEDING TO CURRENT ROW`).

In [5]:
q('''
SELECT user_id, ts, revenue,
       ROUND(SUM(revenue) OVER (PARTITION BY user_id ORDER BY ts
             ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 1) AS running_total,
       ROUND(AVG(revenue) OVER (PARTITION BY user_id ORDER BY ts
             ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 1)         AS mov_avg_3
FROM events
WHERE event = 'buy'
ORDER BY user_id, ts
''')

,user_id,ts,revenue,running_total,mov_avg_3
0,u1,2026-01-03 10:05,45.0,45.0,45.0
1,u1,2026-02-11 21:10,80.0,125.0,62.5
2,u2,2026-01-05 14:30,20.0,20.0,20.0
3,u2,2026-03-02 08:05,120.0,140.0,70.0
4,u3,2026-02-01 12:30,35.0,35.0,35.0
5,u4,2026-01-08 16:10,60.0,60.0,60.0


**The trap to name unprompted:** with **ties in the ORDER BY**, the default frame is `RANGE BETWEEN ...`, which includes *all tied peers* — same timestamp, same running total, doubled sums. Use `ROWS` explicitly when ties exist. Knowing RANGE-vs-ROWS is senior-level signal.

## 4. NTILE and PERCENT_RANK — Bucketing

In [6]:
q('''
SELECT user_id, revenue,
       NTILE(2) OVER (ORDER BY revenue) AS revenue_half,
       ROUND(PERCENT_RANK() OVER (ORDER BY revenue), 2) AS pct_rank
FROM events WHERE event = 'buy'
ORDER BY revenue
''')

,user_id,revenue,revenue_half,pct_rank
0,u2,20.0,1,0.0
1,u3,35.0,1,0.2
2,u1,45.0,1,0.4
3,u4,60.0,2,0.6
4,u1,80.0,2,0.8
5,u2,120.0,2,1.0


`NTILE(k)` makes equal-ish buckets (median = filter `NTILE(2)=1` take max, or `PERCENT_RANK() <= 0.5`); `PERCENT_RANK` is `(rank-1)/(n-1)` ∈ [0,1]. CUME_DIST is the "what fraction is ≤ this value" sibling.

## 5. FIRST_VALUE / LAST_VALUE — and the Frame Gotcha

In [7]:
q('''
SELECT user_id, ts, revenue,
       FIRST_VALUE(revenue) OVER w AS first_purchase,
       LAST_VALUE(revenue)  OVER w AS last_so_far
FROM events
WHERE event = 'buy'
WINDOW w AS (PARTITION BY user_id ORDER BY ts
             ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)
ORDER BY user_id, ts
''')

,user_id,ts,revenue,first_purchase,last_so_far
0,u1,2026-01-03 10:05,45.0,45.0,80.0
1,u1,2026-02-11 21:10,80.0,45.0,80.0
2,u2,2026-01-05 14:30,20.0,20.0,120.0
3,u2,2026-03-02 08:05,120.0,20.0,120.0
4,u3,2026-02-01 12:30,35.0,35.0,35.0
5,u4,2026-01-08 16:10,60.0,60.0,60.0


**The gotcha worth more than the syntax:** `LAST_VALUE` with the *default* frame returns the current row (frame ends at CURRENT ROW), so "latest value" needs the explicit `UNBOUNDED FOLLOWING` — exactly what the cell above does. Interviewers love this one because most candidates get it wrong.

## 6. Sessionization — the Capstone Drill

*"Sessionize events: a new session starts when the gap exceeds 30 minutes."* This is the classic hard screen question. Strategy: **LAG → flag new-session rows → running SUM of flags = session id.** Three named CTE stages, each doing one thing — narrate it that way.

In [8]:
q('''
WITH gaps AS (
  SELECT user_id, ts, event,
         LAG(ts) OVER (PARTITION BY user_id ORDER BY ts) AS prev_ts
  FROM events
),
flagged AS (
  SELECT user_id, ts, event,
         CASE WHEN prev_ts IS NULL
               OR (julianday(ts) - julianday(prev_ts)) * 24 * 60 > 30
              THEN 1 ELSE 0 END AS new_session
  FROM gaps
),
sessionized AS (
  SELECT user_id, ts, event,
         SUM(new_session) OVER (PARTITION BY user_id ORDER BY ts
             ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS session_no
  FROM flagged
)
SELECT user_id, session_no,
       COUNT(*) AS events_in_session,
       MIN(ts) AS session_start, MAX(ts) AS session_end
FROM sessionized
GROUP BY user_id, session_no
ORDER BY user_id, session_start
''')

,user_id,session_no,events_in_session,session_start,session_end
0,u1,1,2,2026-01-03 09:15,2026-01-03 09:20
1,u1,2,2,2026-01-03 10:05,2026-01-03 10:07
2,u1,3,2,2026-02-11 21:00,2026-02-11 21:10
3,u2,1,2,2026-01-05 14:00,2026-01-05 14:30
4,u2,2,2,2026-03-02 08:00,2026-03-02 08:05
5,u3,1,2,2026-01-20 11:00,2026-01-20 11:02
6,u3,2,1,2026-02-01 12:00,2026-02-01 12:00
7,u3,3,1,2026-02-01 12:30,2026-02-01 12:30
8,u4,1,2,2026-01-08 16:00,2026-01-08 16:10
9,u4,2,2,2026-01-09 10:00,2026-01-09 10:01


**Narrate the three stages:** (1) *gaps*: attach the previous timestamp; (2) *flagged*: 1 whenever the gap breaks the 30-minute rule (or it's the first event); (3) *sessionized*: cumulatively sum flags — every session start bumps the counter, giving a dense session id. This pattern (flag + running sum) generalizes to streaks, funnels, and "days since last purchase" features.

## 7. Join Deep-Dive: the Four Outer Behaviors

The joins interviewers draw on the whiteboard — executed with intentional NULL-mismatch and duplicate-key cases so you can *show* the semantics, not recite them:

In [9]:
# duplicate-key join: departments has a dup? make one
conn.execute("INSERT INTO users VALUES ('u1','trial','2025-10-01','IN')")
conn.commit()

q('''
SELECT e.user_id, e.event, u.plan
FROM events e JOIN users u ON e.user_id = u.user_id
WHERE e.event = 'buy'
ORDER BY e.user_id
''')

,user_id,event,plan
0,u1,buy,pro
1,u1,buy,trial
2,u1,buy,pro
3,u1,buy,trial
4,u2,buy,free
5,u2,buy,free
6,u3,buy,pro
7,u4,buy,free


**What just happened:** u1 exists **twice** in `users` (pro + trial rows) — an inner join with a duplicated key **multiplies** u1's buy rows. This is the row-explosion bug behind most "my joined table has more rows than expected" production incidents. Defenses to recite: check key uniqueness first (`GROUP BY key HAVING COUNT(*) > 1`), or use `QUALIFY ROW_NUMBER() ... = 1` / dedupe before joining. u4 has no matching extra row and nothing multiplies — say why: duplication must be on the *many* side to explode.

In [10]:
# full outer join emulation in SQLite (LEFT JOIN UNION RIGHT-emulation)
q('''
SELECT u.user_id, u.plan, COUNT(e.event) AS n_events
FROM users u
LEFT JOIN events e ON e.user_id = u.user_id
GROUP BY u.user_id, u.plan
ORDER BY u.user_id
''')

,user_id,plan,n_events
0,u1,pro,6
1,u1,trial,6
2,u2,free,4
3,u3,pro,4
4,u4,free,4


## 8. The Pandas Translation Table

Screens often accept either tool — showing the mapping proves you understand both. Rehearse the equivalence:

In [11]:
ev = pd.read_sql_query("SELECT * FROM events", conn)

# LAG within group
ev["prev_ts"] = ev.groupby("user_id")["ts"].shift(1)

# running total within group
ev["running_rev"] = (ev.sort_values("ts")
                       .groupby("user_id")["revenue"]
                       .transform(lambda s: s.fillna(0).cumsum()))

# ROW_NUMBER within group
ev["rn"] = (ev.sort_values("ts")
              .groupby("user_id").cumcount() + 1)

# top-1 per group
top = (ev.sort_values("revenue", ascending=False)
         .groupby("user_id", as_index=False).head(1))

print(ev[ev.user_id == "u1"][["ts", "event", "prev_ts", "running_rev", "rn"]].to_string(index=False))
print(); print("top purchase per user:"); print(top[["user_id", "event", "revenue"]].to_string(index=False))

              ts event          prev_ts  running_rev  rn
2026-01-03 09:15 login              NaN          0.0   1
2026-01-03 09:20  view 2026-01-03 09:15          0.0   2
2026-01-03 10:05   buy 2026-01-03 09:20         45.0   3
2026-01-03 10:07  view 2026-01-03 10:05         45.0   4
2026-02-11 21:00 login 2026-01-03 10:07         45.0   5
2026-02-11 21:10   buy 2026-02-11 21:00        125.0   6

top purchase per user:
user_id event  revenue
     u2   buy    120.0
     u1   buy     80.0
     u4   buy     60.0
     u3   buy     35.0


| SQL window | pandas idiom |
| --- | --- |
| `ROW_NUMBER() OVER (PARTITION BY g ORDER BY v)` | `df.sort_values("v").groupby("g").cumcount() + 1` |
| `LAG(x) OVER (PARTITION BY g ORDER BY t)` | `df.groupby("g")["x"].shift(1)` |
| `SUM(x) OVER (... ROWS UNBOUNDED PRECEDING)` | `df.groupby("g")["x"].cumsum()` |
| `AVG(x) OVER (... 2 PRECEDING)` | `df.groupby("g")["x"].transform(lambda s: s.rolling(3).mean())` |
| top-1 per group | `sort_values().groupby().head(1)` |
| `NTILE`, `PERCENT_RANK` | `pd.qcut` / `rank(pct=True)` |

## 9. The Rehearsal Set — Write These Cold

Time yourself: 5 minutes each, on this schema.

1. Each user's **second** purchase → `ROW_NUMBER` + `rn = 2`.
2. Users whose **latest** event was a `buy` → `ROW_NUMBER ... ORDER BY ts DESC` + `rn=1` + `event='buy'`.
3. 7-day trailing revenue per user → frame `ROWS BETWEEN 6 PRECEDING AND CURRENT ROW` (or RANGE on dates in engines with interval support).
4. Percentage of each user's revenue by event → `SUM(...) OVER (PARTITION BY user_id)` as denominator.
5. **Consecutive-login streaks** → flag + running-sum pattern from sessionization, grouped by streak id.
6. Dedupe users keeping the **most recent** plan per user → `ROW_NUMBER() ... ORDER BY signup_date DESC` + `rn=1` (the fix for the row-explosion demo above).